In [1]:
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from models import *
batch_size = 32
seed = 42
lr = 3e-4
epochs = 15
torch.use_deterministic_algorithms(True, warn_only=True)
device = 'mps'

In [2]:
def val_evaluate(n, file, sorted=True, fc=True, dist=True, avg=False):
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = models[n]().to(device) if sorted else NeuralNetUnsorted4().to(device)
    if not fc: model = NeuralNetNoFC4().to(device)
    if not dist: model = NeuralNetNoDist4().to(device)
    if avg: model = NeuralNetAvg4().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCEWithLogitsLoss()
    correct = 0
    total_loss = 0.0
    total = 0
    val_inputs = np.memmap(f'val_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    val_targets = np.memmap(f'val_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    dataset = TensorDataset(torch.from_numpy(val_inputs), torch.from_numpy(val_targets))
    val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for batch_inputs, batch_targets in val_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            logits = model(*inputs)
            loss = loss_function(logits.squeeze(), batch_targets)
            total_loss += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            predictions = logits.squeeze() > 0
            correct += (predictions == batch_targets).sum().item()
    print("Validation loss:", total_loss / total)
    print("Accuracy:", correct / total)

In [3]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), 
    transforms.ToTensor()
])

In [4]:
def train_model(n, sorted=True, fc=True, dist=True, avg=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = models[n]().to(device) if sorted else NeuralNetUnsorted4().to(device)
    if not fc: model = NeuralNetNoFC4().to(device)
    if not dist: model = NeuralNetNoDist4().to(device)
    if avg: model = NeuralNetAvg4().to(device)
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    inputs_mmap = np.memmap(f'train_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(200000, n, 128, 128, 3))
    targets_mmap = np.memmap(f'train_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(200000,))
    dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_inputs, batch_targets in train_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            inputs = [torch.stack([train_transforms(img) for img in input]).to(device) for input in inputs]
            optimizer.zero_grad()
            preds = model(*inputs)
            loss = loss_function(preds.squeeze(), batch_targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1} avg loss = {running_loss / len(train_loader)}')
        file = f'model_{n}{'_unsorted' if not sorted else ""}{"" if fc else "_nofc"}{"" if dist else "_nodist"}{"_avg" if avg else ""}/epoch_{epoch+1}.pth'
        torch.save(model.state_dict(), file)
        val_evaluate(n, file, sorted=sorted, fc=fc, dist=dist, avg=avg)

In [ ]:
train_model(2)

Epoch 1 avg loss = 0.5618780746030807
Validation loss: 0.44087655193805697
Accuracy: 0.7895
Epoch 2 avg loss = 0.4535959800386429
Validation loss: 0.3977194437980652
Accuracy: 0.81745
Epoch 3 avg loss = 0.4175688469529152
Validation loss: 0.3719407778263092
Accuracy: 0.8293
Epoch 4 avg loss = 0.39762348620176313
Validation loss: 0.3568021329879761
Accuracy: 0.83805
Epoch 5 avg loss = 0.3830601697325706
Validation loss: 0.34628455262184143
Accuracy: 0.8405
Epoch 6 avg loss = 0.37184887986421583
Validation loss: 0.3345809209704399
Accuracy: 0.84865
Epoch 7 avg loss = 0.362693281481266
Validation loss: 0.3317284345507622
Accuracy: 0.84955
Epoch 8 avg loss = 0.35489523059129713
Validation loss: 0.32167112290859223
Accuracy: 0.8571
Epoch 9 avg loss = 0.34854189498782157
Validation loss: 0.3259606399178505
Accuracy: 0.8548
Epoch 10 avg loss = 0.33952800695896146
Validation loss: 0.315718441092968
Accuracy: 0.85925
Epoch 11 avg loss = 0.33537550275444983
Validation loss: 0.31282790454626086
A

In [ ]:
train_model(3)

Epoch 1 avg loss = 0.5000663499188424
Validation loss: 0.38194017713069917
Accuracy: 0.8205
Epoch 2 avg loss = 0.3909498315215111
Validation loss: 0.33141691472530366
Accuracy: 0.8488
Epoch 3 avg loss = 0.35417938549757005
Validation loss: 0.31033872302770615
Accuracy: 0.85965
Epoch 4 avg loss = 0.33559416424274446
Validation loss: 0.2932952523112297
Accuracy: 0.8711
Epoch 5 avg loss = 0.3216740989089012
Validation loss: 0.2894928816139698
Accuracy: 0.87445
Epoch 6 avg loss = 0.3131670376157761
Validation loss: 0.27923602546453474
Accuracy: 0.876
Epoch 7 avg loss = 0.3024762105178833
Validation loss: 0.2738050197482109
Accuracy: 0.8812
Epoch 8 avg loss = 0.2926400558686256
Validation loss: 0.2724926851868629
Accuracy: 0.8805
Epoch 9 avg loss = 0.286744461414814
Validation loss: 0.27512111632823943
Accuracy: 0.8774
Epoch 10 avg loss = 0.2795292133319378
Validation loss: 0.26375015823841097
Accuracy: 0.8839
Epoch 11 avg loss = 0.2742170989501476
Validation loss: 0.2577274885892868
Accura

In [ ]:
train_model(4)

/var/folders/_b/yynhlh9s339c2gvwx12flcqm0000gp/T/ipykernel_33433/1605132405.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic 

Epoch 1 avg loss = 0.48497579655647277
Validation loss: 0.36097163441181185
Accuracy: 0.83615
Epoch 2 avg loss = 0.36844899152040483
Validation loss: 0.31148729357719424
Accuracy: 0.86225
Epoch 3 avg loss = 0.3320790645337105
Validation loss: 0.285603875374794
Accuracy: 0.8742
Epoch 4 avg loss = 0.3108917411720753
Validation loss: 0.26949146897792814
Accuracy: 0.88325
Epoch 5 avg loss = 0.29514660343885424
Validation loss: 0.26036875136494636
Accuracy: 0.8845
Epoch 6 avg loss = 0.28329068779587746
Validation loss: 0.2583717037975788
Accuracy: 0.88945
Epoch 7 avg loss = 0.27176145117640493
Validation loss: 0.2607308459162712
Accuracy: 0.88925
Epoch 8 avg loss = 0.2638983999466896
Validation loss: 0.24374641728997232
Accuracy: 0.8942
Epoch 9 avg loss = 0.25607268990516663
Validation loss: 0.24344644355773926
Accuracy: 0.89255
Epoch 10 avg loss = 0.24932232737481594
Validation loss: 0.23474652547836303
Accuracy: 0.8967
Epoch 11 avg loss = 0.2426542980194092
Validation loss: 0.234886074411

In [ ]:
train_model(5)

Epoch 1 avg loss = 0.4538509310388565
Validation loss: 0.33845446763038634
Accuracy: 0.84765
Epoch 2 avg loss = 0.34987962484836577
Validation loss: 0.2983224333047867
Accuracy: 0.867
Epoch 3 avg loss = 0.3170448440289497
Validation loss: 0.27001417073011397
Accuracy: 0.8828
Epoch 4 avg loss = 0.2961599120283127
Validation loss: 0.25760829465985297
Accuracy: 0.8872
Epoch 5 avg loss = 0.2830990895330906
Validation loss: 0.2553681345164776
Accuracy: 0.8914
Epoch 6 avg loss = 0.26999779222130776
Validation loss: 0.2460605349421501
Accuracy: 0.8921
Epoch 7 avg loss = 0.26248529034137724
Validation loss: 0.23480144137144088
Accuracy: 0.89945
Epoch 8 avg loss = 0.25345640485525134
Validation loss: 0.23538209129571916
Accuracy: 0.89855
Epoch 9 avg loss = 0.2462835614824295
Validation loss: 0.22577909673452376
Accuracy: 0.90305
Epoch 10 avg loss = 0.23876225294947626
Validation loss: 0.22641289055347444
Accuracy: 0.90195
Epoch 11 avg loss = 0.23308804187834262
Validation loss: 0.22750056032538

In [ ]:
train_model(6)

Epoch 1 avg loss = 0.45082406564474103
Validation loss: 0.32177415531873704
Accuracy: 0.85855
Epoch 2 avg loss = 0.3391845682275295
Validation loss: 0.2769051476120949
Accuracy: 0.88025
Epoch 3 avg loss = 0.30440655704617503
Validation loss: 0.2562427994251251
Accuracy: 0.88935
Epoch 4 avg loss = 0.2847293627882004
Validation loss: 0.24339157814383508
Accuracy: 0.898
Epoch 5 avg loss = 0.2722532246899605
Validation loss: 0.22959382878541945
Accuracy: 0.9033
Epoch 6 avg loss = 0.25857795790553095
Validation loss: 0.2216962286055088
Accuracy: 0.9076
Epoch 7 avg loss = 0.24699250956475735
Validation loss: 0.2161819361180067
Accuracy: 0.9092
Epoch 8 avg loss = 0.2399233068239689
Validation loss: 0.22130458107590675
Accuracy: 0.9078
Epoch 9 avg loss = 0.23181355821847915
Validation loss: 0.21530421146154405
Accuracy: 0.9109
Epoch 10 avg loss = 0.2289355550378561
Validation loss: 0.21060846858024598
Accuracy: 0.91285
Epoch 11 avg loss = 0.21986066066741944
Validation loss: 0.2058252968966961

In [ ]:
train_model(4, sorted=False)

In [10]:
train_model(4, fc=False)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1 avg loss = 0.5314943604469299
Validation loss: 0.4612032893896103
Accuracy: 0.77965
Epoch 2 avg loss = 0.4140485180997848
Validation loss: 0.4002522472858429
Accuracy: 0.8215
Epoch 3 avg loss = 0.37159153536438944
Validation loss: 0.3669195780992508
Accuracy: 0.8369
Epoch 4 avg loss = 0.3459937042438984
Validation loss: 0.3624360492706299
Accuracy: 0.8394
Epoch 5 avg loss = 0.32845966583848
Validation loss: 0.33293715121746065
Accuracy: 0.8547
Epoch 6 avg loss = 0.3133250719630718
Validation loss: 0.33238415157794954
Accuracy: 0.85675
Epoch 7 avg loss = 0.30170398923516273
Validation loss: 0.32841959565877915
Accuracy: 0.8588
Epoch 8 avg loss = 0.2919392024171352
Validation loss: 0.32748848696947097
Accuracy: 0.85795
Epoch 9 avg loss = 0.28330166808724405
Validation loss: 0.32246203281879426
Accuracy: 0.8609
Epoch 10 avg loss = 0.2771967292356491
Validation loss: 0.3255654548883438
Accuracy: 0.85975
Epoch 11 avg loss = 0.26991051884412764
Validation loss: 0.3112670743346214
Acc

In [5]:
train_model(4, dist=False)

/var/folders/_b/yynhlh9s339c2gvwx12flcqm0000gp/T/ipykernel_33956/2098405739.py:14: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))


Epoch 1 avg loss = 0.6255571859455109
Validation loss: 0.5113537139415741
Accuracy: 0.75155
Epoch 2 avg loss = 0.48636680683374406
Validation loss: 0.41955143756866453
Accuracy: 0.80695
Epoch 3 avg loss = 0.4536028500390053
Validation loss: 0.3974159104347229
Accuracy: 0.81725
Epoch 4 avg loss = 0.4341879276919365
Validation loss: 0.38462968673706055
Accuracy: 0.8288
Epoch 5 avg loss = 0.41899008999109266
Validation loss: 0.37091738283634185
Accuracy: 0.83115
Epoch 6 avg loss = 0.40816066131830214
Validation loss: 0.35610857081413266
Accuracy: 0.8428
Epoch 7 avg loss = 0.3936987122440338
Validation loss: 0.3437605145931244
Accuracy: 0.84905
Epoch 8 avg loss = 0.3812756669998169
Validation loss: 0.34536667129993437
Accuracy: 0.84615
Epoch 9 avg loss = 0.373721717569828
Validation loss: 0.3231107752203941
Accuracy: 0.8599
Epoch 10 avg loss = 0.36511497864127157
Validation loss: 0.31608190598487856
Accuracy: 0.86285
Epoch 11 avg loss = 0.3601162183332443
Validation loss: 0.309743372631073

In [6]:
train_model(4, avg=True)

Epoch 1 avg loss = 0.4749715756225586
Validation loss: 0.36288773493766785
Accuracy: 0.83825
Epoch 2 avg loss = 0.3696877606678009
Validation loss: 0.31934749438762666
Accuracy: 0.8598
Epoch 3 avg loss = 0.3347955203962326
Validation loss: 0.29259013628959657
Accuracy: 0.86945
Epoch 4 avg loss = 0.3122981085932255
Validation loss: 0.2724342405080795
Accuracy: 0.88315
Epoch 5 avg loss = 0.29594096870541575
Validation loss: 0.26853879789114
Accuracy: 0.88525
Epoch 6 avg loss = 0.28397415288329125
Validation loss: 0.2566398638248444
Accuracy: 0.89235
Epoch 7 avg loss = 0.2738580516457558
Validation loss: 0.2508507341504097
Accuracy: 0.89315
Epoch 8 avg loss = 0.26664471912801263
Validation loss: 0.25526606295108795
Accuracy: 0.8915
Epoch 9 avg loss = 0.2596461782026291
Validation loss: 0.2443982764840126
Accuracy: 0.89625
Epoch 10 avg loss = 0.251099808447361
Validation loss: 0.23936747321486473
Accuracy: 0.8996
Epoch 11 avg loss = 0.24636816669285297
Validation loss: 0.24468754937648773
